## Act database scraper (indiacode.gov.in)
The site is a JavaScript app, but the table on `/act/{id}/details` is filled from a JSON API. Each act's metadata comes back in the same list call, so no need to open 800+ detail pages. Fields an act lacks are simply absent, and pandas turns the gaps into blanks.

In [ ]:
import time
import requests
import pandas as pd

BASE_URL = "https://indiacode.gov.in"
COLLECTION_ID = "69a0c1fb-7b22-4481-b16a-1dc59b5d02e6"
OBJECTS_API = f"{BASE_URL}/server/api/discover/search/objects"

# System/bookkeeping fields you probably don't want as columns
DROP_KEYS = {"dc.date.accessioned", "dc.date.available", "dc.description.provenance"}

# Fill in after you see the coverage table: raw metadata key -> column name you want
LABELS = {
    "dc.title": "Act",
    "dc.date.act_year": "Act Year",
    "dc.date.enact_date": "Enactment Date",
}

SEP = " | "  # joins multi-valued fields (e.g. several ministries)


def get_json(session, url, params, retries=4):
    for attempt in range(retries):
        try:
            r = session.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            wait = 2 ** attempt
            print(f"  retry {attempt + 1}/{retries} after error: {e} (sleep {wait}s)")
            time.sleep(wait)
    raise RuntimeError(f"Failed after {retries} attempts: {url} {params}")


def fetch_all_items(session, page_size=100, delay=0.5):
    """Return the raw 'indexableObject' dict of every act in the collection."""
    items, page, total_pages = [], 0, None
    while total_pages is None or page < total_pages:
        data = get_json(session, OBJECTS_API, {
            "scope": COLLECTION_ID, "configuration": "collection",
            "page": page, "size": page_size, "sort": "dc.date.issued,DESC",
        })
        sr = data.get("_embedded", {}).get("searchResult", {})
        total_pages = sr.get("page", {}).get("totalPages", total_pages or 1)
        objs = sr.get("_embedded", {}).get("objects", [])
        for o in objs:
            io = o.get("_embedded", {}).get("indexableObject")
            if io:
                items.append(io)
        print(f"page {page + 1}/{total_pages}: {len(objs)} acts (running total {len(items)})")
        page += 1
        time.sleep(delay)
    return items


def flatten_item(io):
    """One act -> one flat dict. Only keys the act actually has are present."""
    uuid = io.get("uuid") or io.get("id")
    row = {
        "uuid": uuid,
        "act_url": f"{BASE_URL}/act/{uuid}",
        "details_url": f"{BASE_URL}/act/{uuid}/details",
        "handle": io.get("handle"),
        "dc.title": io.get("name"),
    }
    for key, vals in (io.get("metadata") or {}).items():
        if key in DROP_KEYS:
            continue
        clean = [str(v.get("value")).strip() for v in vals
                 if v.get("value") not in (None, "")]
        if clean:                                   # skip empty fields entirely
            row[key] = SEP.join(clean)
    return row


def build_dataframe(items, labels=LABELS):
    df = pd.DataFrame([flatten_item(i) for i in items])   # union of all keys; gaps -> NaN
    id_cols = ["uuid", "act_url", "details_url", "handle"]
    other = [c for c in df.columns if c not in id_cols]
    other.sort(key=lambda c: -df[c].notna().sum())         # most-complete columns first
    df = df[id_cols + other]
    return df.rename(columns=labels)


def coverage(df):
    c = df.notna().sum().rename("filled").to_frame()
    c["pct"] = (100 * c["filled"] / len(df)).round(1)
    return c.sort_values("filled", ascending=False)


def filter_years(df, lo=2010, hi=2026, date_cols=("dc.date.issued", "Enactment Date", "Act Year")):
    """Client-side version of f.dateIssued.min/max from your URL."""
    year = pd.Series(pd.NA, index=df.index, dtype="object")
    for col in date_cols:
        if col in df.columns:
            year = year.fillna(df[col].astype("string").str.extract(r"(\d{4})")[0])
    year = pd.to_numeric(year, errors="coerce")
    return df[year.between(lo, hi)].reset_index(drop=True)


### Step 1: pull every act (list + all metadata in one pass)

In [ ]:
session = requests.Session()
session.headers.update({'User-Agent': 'Mozilla/5.0', 'Accept': 'application/json'})
items = fetch_all_items(session)
print(len(items), 'acts fetched')

### Step 2: inspect which fields exist and how often
Use this table to fill in the `LABELS` dict at the top (raw key -> the column name shown on the site), then re-run Step 3.

In [ ]:
df = build_dataframe(items)
display(coverage(df))

### Step 3: optional year filter (your URL used 2010 to 2026), then save

In [ ]:
df_final = filter_years(df, 2010, 2026)   # skip this line to keep everything
print(df_final.shape)
df_final.to_csv('indiacode_acts_details.csv', index=False, encoding='utf-8-sig')
df_final.head()

### Optional check: compare against the rendered 'Act Details' table for one act
Only needed if Step 2 is missing a field you can see on the website. Needs `pip install playwright` and `playwright install chromium` (both free). The selector is a guess, so adjust it if the table is not found.

In [ ]:
# from playwright.sync_api import sync_playwright
# def scrape_details_page(url):
#     with sync_playwright() as p:
#         b = p.chromium.launch(); pg = b.new_page()
#         pg.goto(url, wait_until="networkidle"); pg.wait_for_selector("table", timeout=20000)
#         out = {}
#         for tr in pg.query_selector_all("table tr"):
#             cells = [c.inner_text().strip() for c in tr.query_selector_all("th, td")]
#             if len(cells) >= 2 and cells[0]:
#                 out[cells[0]] = " | ".join(cells[1:])
#         b.close(); return out
# scrape_details_page(df.loc[0, "details_url"])